# GPT-5.4-mini(API)를 활용한 뉴스 이진 코딩 분석

**News content binary coding tool**

연구자가 정의한 항목에 해당하는지를 예(1) / 아니오(0)로 판단합니다. `ANALYSIS_CRITERIA`와 `PROMPT_TEMPLATE`의 **[대괄호]** 부분을 본인의 연구 주제에 맞게 교체한 뒤 실행하세요.

---

## 실행 환경 요약

- 모델: `gpt-5.4-mini` / API: OpenAI **Responses API** (`client.responses.create`) + JSON Schema 구조화 출력(`strict: true`)
- `reasoning_effort="none"` (분류·추출·요약 과제에 적합). `temperature`는 `reasoning_effort="none"`일 때만 지정할 수 있습니다.
- 이전 버전: [GPT-4.1 버전 저장소](https://github.com/HanJunLeeKR/News_Content_Analysis_with_GPT-4.1) (Chat Completions + `json_object`)
- 두 버전의 차이는 저장소 `README.md`의 「GPT-4.1 버전과의 차이」를 참고하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Colab/Jupyter용). 로컬 환경에서는 저장소의 requirements.txt 사용:
#   pip install -r requirements.txt
# ═══════════════════════════════════════════════════════════════════════
%pip install --upgrade openai pandas tqdm nest-asyncio


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 이진 코딩 분석기 (GPT-5.4-mini 버전)
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Optional, Tuple, Dict
import re
from tqdm import tqdm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. 분석 기준 정의(맥락 제공, 기준 설명, 판단 기준, 예시 포함, LLM을 교육하는 용도로 생각)
# ═════════════════════════════════════════════════════════════════════════════════
# 아래 [대괄호] 부분을 본인의 연구 주제에 맞게 채워 넣으세요.
ANALYSIS_CRITERIA = """
**[코딩 대상 항목] 판단 기준:**

이 분석은 기사가 "[코딩 대상 항목]"에 해당하는지 판단합니다.

**배경:**
- [분석 맥락과 배경을 설명하세요]
- [코딩 대상 항목의 정의를 설명하세요]

**"예(1)"로 판단하는 경우 (다음 중 하나라도 해당):**

1. **[판단 기준 1]**
   - [세부 기준 또는 예시]
   - [세부 기준 또는 예시]

2. **[판단 기준 2]**
   - [세부 기준 또는 예시]
   - [세부 기준 또는 예시]

3. **[판단 기준 3]**
   - [세부 기준 또는 예시]
   - [세부 기준 또는 예시]

**중요:** 기사 본문에서 직접 언급하거나, 정보원(전문가, 정치인, 학자 등)의 발언을 인용하여 간접적으로 언급해도 "예(1)"로 판단

**"아니오(0)"로 판단하는 경우:**
- 단순 사실 전달만 하는 경우
- [코딩 대상 항목]과 무관한 내용
"""


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """Determine whether the following Korean news article meets the coding criterion: "[코딩 대상 항목]".

[NEWS ARTICLE]:
{text}

{criteria}

Instructions:
1. Identify expressions related to "[코딩 대상 항목]" in the article.
2. Check for both direct statements and indirect quotes from sources.
3. Determine whether any of the conditions in the criteria above are satisfied.
4. Select 1 (yes) or 0 (no) according to the criteria above.
5. Provide specific evidence from the article text.
6. Rate confidence as an integer between 1 and 100.

Output ONLY valid JSON. Do not add fields not in the schema."""


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. JSON 파서
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""

    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")
        response_text = response_text.strip()

        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass

        for pattern in [r'```json\s*\n?(.*?)\n?```', r'```\s*\n?(.*?)\n?```']:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                try:
                    return json.loads(match.group(1).strip())
                except json.JSONDecodeError:
                    continue

        for pattern in [r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', r'\{(?:[^{}]|{[^{}]*})*\}']:
            for match in re.findall(pattern, response_text):
                try:
                    result = json.loads(match)
                    if EnhancedJSONParser._validate_result(result):
                        return result
                except json.JSONDecodeError:
                    continue

        extracted_data = {}

        for p in [r'"coding_result"\s*:\s*([01])', r'coding_result["\']?\s*[:=]\s*([01])']:
            m = re.search(p, response_text, re.IGNORECASE)
            if m:
                extracted_data['coding_result'] = int(m.group(1))
                break

        for p in [r'"reason"\s*:\s*"([^"]*(?:\\.[^"]*)*)"']:
            m = re.search(p, response_text, re.IGNORECASE | re.DOTALL)
            if m:
                extracted_data['reason'] = m.group(1).strip()
                break

        for p in [r'"confidence"\s*:\s*([0-9]+)']:
            m = re.search(p, response_text, re.IGNORECASE)
            if m:
                try:
                    extracted_data['confidence'] = int(m.group(1))
                except ValueError:
                    pass
                break

        if extracted_data and EnhancedJSONParser._validate_result(extracted_data):
            return extracted_data
        return EnhancedJSONParser._get_default_result("파싱 실패")

    @staticmethod
    def _validate_result(result: Dict) -> bool:
        return (
            isinstance(result, dict) and
            'coding_result' in result and
            isinstance(result['coding_result'], int) and
            result['coding_result'] in [0, 1]
        )

    @staticmethod
    def _get_default_result(error_msg: str) -> Dict:
        return {
            'coding_result': -1,
            'reason': f'분석 실패: {error_msg}',
            'confidence': 0
        }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 이진 코딩 분석기 (GPT-5.4-mini 버전)
# ═══════════════════════════════════════════════════════════════════════
class BinaryCodingAnalyzer:
    """이진 코딩 분석기 (GPT-5.4-mini용)"""

    def __init__(self, api_key, model, output_dir='.',
                 concurrent_requests=2, rate_limit_delay=1.0, batch_size=5,
                 reasoning_effort='none', text_verbosity='medium',
                 temperature: Optional[float] = None,
                 debug=False):
        # ── GPT-5.4-mini: temperature는 reasoning_effort='none'일 때만 유효 ──
        if temperature is not None and reasoning_effort != 'none':
            raise ValueError(
                f"temperature is only valid when reasoning_effort='none'. "
                f"Got reasoning_effort='{reasoning_effort}'."
            )
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.reasoning_effort = reasoning_effort
        self.text_verbosity = text_verbosity
        self.temperature = temperature
        self.debug = debug
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)
        self.parsing_successes = 0
        self.parsing_failures = 0
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)
        os.makedirs(self.output_dir, exist_ok=True)

    def combine_title_text(self, title: str, text: str) -> str:
        if pd.isna(title): title = ""
        if pd.isna(text): text = ""
        title = str(title).strip()
        text = str(text).strip()
        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:10000]

    async def analyze_single_article(self, title: str, text: str) -> Tuple[int, str, int, float]:
        async with self.semaphore:
            try:
                combined_text = self.combine_title_text(title, text)
                if not combined_text:
                    return -1, "빈 텍스트", 0, 0.0

                prompt = PROMPT_TEMPLATE.format(
                    text=combined_text,
                    criteria=ANALYSIS_CRITERIA
                )

                # ═══════════════════════════════════════════════════
                # GPT-5.4-mini API 호출
                # reasoning_effort options: none | low | medium | high | xhigh
                # temperature: only valid when reasoning_effort='none'
                # ═══════════════════════════════════════════════════
                api_params = dict(
                    model=self.model,
                    input=[{"role": "user", "content": prompt}],
                    max_output_tokens=1024,
                    reasoning={"effort": self.reasoning_effort},
                    text={
                        "verbosity": self.text_verbosity,
                        "format": {
                            "type": "json_schema",
                            "name": "binary_coding",
                            "strict": True,
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "coding_result": {
                                        "type": "integer",
                                        "description": "Does the article meet the coding criterion? (1=yes, 0=no)",
                                        "enum": [0, 1]
                                    },
                                    "reason": {
                                        "type": "string",
                                        "description": "Specific evidence from the article text or source quotes"
                                    },
                                    "confidence": {
                                        "type": "integer",
                                        "description": "Confidence score (1-100)",
                                        "minimum": 1,
                                        "maximum": 100
                                    }
                                },
                                "required": ["coding_result", "reason", "confidence"],
                                "additionalProperties": False
                            }
                        }
                    }
                )
                if self.temperature is not None:
                    api_params["temperature"] = self.temperature

                response = await self.client.responses.create(**api_params)

                if hasattr(response, 'output_text') and response.output_text:
                    response_text = response.output_text
                else:
                    response_text = ""
                    for item in response.output:
                        if item.type == "message":
                            for content in item.content:
                                if hasattr(content, 'text'):
                                    response_text += content.text

                response_text = response_text.strip()
                if not response_text:
                    return -1, "빈 응답", 0, 0.0

                result = EnhancedJSONParser.extract_json_robust(response_text)
                coding_result = result.get('coding_result', -1)
                reason         = result.get('reason', '파싱 실패')
                confidence     = result.get('confidence', 0)

                if coding_result not in [0, 1]:
                    coding_result = -1
                try:
                    confidence = int(confidence)
                    confidence = max(0, min(100, confidence))
                except (ValueError, TypeError):
                    confidence = 0

                if coding_result in [0, 1]:
                    self.parsing_successes += 1
                else:
                    self.parsing_failures += 1

                input_tokens  = response.usage.input_tokens if hasattr(response.usage, 'input_tokens') else 0
                output_tokens = response.usage.output_tokens if hasattr(response.usage, 'output_tokens') else 0
                estimated_cost = (input_tokens * PRICE_INPUT_PER_1M / 1_000_000) + (output_tokens * PRICE_OUTPUT_PER_1M / 1_000_000)
                self.total_cost += estimated_cost

                return coding_result, reason, confidence, estimated_cost

            except Exception as e:
                if self.debug:
                    print(f"[DEBUG] Exception: {type(e).__name__}: {str(e)}")
                self.parsing_failures += 1
                return -1, f"오류: {str(e)[:50]}", 0, 0.0
            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list, list, list]:
        tasks = [
            self.analyze_single_article(row['title'], row['text'])
            for _, row in batch_df.iterrows()
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        warnings, reasons, confidences, costs = [], [], [], []
        for result in results:
            if isinstance(result, Exception):
                warnings.append(-1)
                reasons.append(f"배치 오류: {str(result)[:50]}")
                confidences.append(0)
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                warnings.append(result[0])
                reasons.append(result[1])
                confidences.append(result[2])
                costs.append(result[3])
        return warnings, reasons, confidences, costs

    async def analyze_dataframe(self, df, output_filename="binary_coding_results.csv"):
        output_path = os.path.join(self.output_dir, output_filename)
        all_warnings, all_reasons, all_confidences, all_costs = [], [], [], []
        start_time = time.time()
        pbar = tqdm(total=len(df), desc="이진 코딩 분석 중", unit="건")

        for i in range(0, len(df), self.batch_size):
            batch_df = df.iloc[i:min(i + self.batch_size, len(df))]
            try:
                warnings, reasons, confidences, costs = await self.process_batch(batch_df)
                all_warnings.extend(warnings)
                all_reasons.extend(reasons)
                all_confidences.extend(confidences)
                all_costs.extend(costs)
                pbar.update(len(batch_df))

                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    n = len(df)
                    temp_df = df.copy()
                    temp_df['코딩결과'] = all_warnings + [-1] * (n - len(all_warnings))
                    temp_df['근거'] = all_reasons + [''] * (n - len(all_reasons))
                    temp_df['신뢰도'] = all_confidences + [0] * (n - len(all_confidences))
                    temp_df['비용'] = all_costs + [0.0] * (n - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)
            except Exception as e:
                sz = len(batch_df)
                all_warnings.extend([-1] * sz)
                all_reasons.extend([f"배치 오류: {str(e)[:30]}"] * sz)
                all_confidences.extend([0] * sz)
                all_costs.extend([0.0] * sz)
                self.parsing_failures += sz
                pbar.update(sz)

        pbar.close()
        result_df = df.copy()
        result_df['코딩결과'] = all_warnings
        result_df['근거'] = all_reasons
        result_df['신뢰도'] = all_confidences
        result_df['비용'] = all_costs
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)
        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        total_count   = len(result_df)
        valid_results = result_df[result_df['코딩결과'].isin([0, 1])]
        yes_count     = (valid_results['코딩결과'] == 1).sum()
        no_count      = (valid_results['코딩결과'] == 0).sum()
        fail_count    = (result_df['코딩결과'] == -1).sum()
        print(f"\n{'='*60}")
        print(f"✅ 이진 코딩 분석 완료! (GPT-5.4-mini)")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 분석 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 분석 결과:")
        if len(valid_results) > 0:
            print(f"   • 예(1) - 위기 제기함: {yes_count}건 ({(yes_count/len(valid_results))*100:.1f}%)")
            print(f"   • 아니오(0) - 위기 제기 안함: {no_count}건 ({(no_count/len(valid_results))*100:.1f}%)")
        if fail_count > 0:
            print(f"   • 분석 실패: {fail_count}건")
        print(f"\n📈 분석 성공률:")
        success_rate = (self.parsing_successes / total_count) * 100 if total_count > 0 else 0
        print(f"   • 성공: {self.parsing_successes}건 ({success_rate:.1f}%)")
        print(f"   • 실패: {self.parsing_failures}건 ({(self.parsing_failures/total_count)*100:.1f}%)")
        print(f"{'='*60}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════
# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-5.4-mini"          # GPT-5.4-mini
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")  # 환경변수 사용 권장. 키를 코드에 직접 적은 채 커밋하지 마세요.
INPUT_CSV_PATH  = "Input CSV file path here"
OUTPUT_CSV_PATH = "Output CSV file path here"

# GPT-5.4-mini 전용 설정
# reasoning_effort options: none | low | medium | high | xhigh
# ※ binary coding 태스크에는 'none' 권장
REASONING_EFFORT = "none"
TEXT_VERBOSITY   = "medium"     # low | medium | high
# temperature: reasoning_effort='none'일 때만 유효 (None이면 비활성화)
TEMPERATURE      = None
# ※ GPT-4.1 버전은 temperature=0.0 고정이었습니다. 동일한 재현성이 필요하면 0.0으로 설정하세요(reasoning_effort="none"일 때만 유효).
DEBUG_MODE       = False

# 비용 추정용 단가 (USD / 1M tokens). OpenAI 공식 가격표 기준 gpt-5.4-mini 입력 $0.75, 출력 $4.50 (2026-11 확인). 가격이 바뀌면 수정하세요.
PRICE_INPUT_PER_1M  = 0.75
PRICE_OUTPUT_PER_1M = 4.50
# ──────────────────────────────────────────────────────────────────────

async def main():
    try:
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return

        output_dir  = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)

        analyzer = BinaryCodingAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir,
            reasoning_effort=REASONING_EFFORT,
            text_verbosity=TEXT_VERBOSITY,
            temperature=TEMPERATURE,
            debug=DEBUG_MODE
        )
        await analyzer.analyze_dataframe(df, output_file)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")
        import traceback; traceback.print_exc()

def run_analysis():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    asyncio.run(main())

if __name__ == "__main__":
    run_analysis()